<a href="https://colab.research.google.com/github/richarddzh/airbnb_mle/blob/main/colab/tinylmzhdata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prepare Data Chinese

In [8]:
# ==================== 需要修改的配置 ====================
DEST_REPO = "richarddzh/chinese-small-lm-corpus"  # 改成你的 用户名/数据集名
SHARD_ROWS = 50_000                                # 每个 Parquet 分片的行数
WRITE_BATCH_ROWS = 2_000                           # 内存中的写入批次
# ======================================================

%pip install -q -U "datasets>=3.0" "huggingface_hub>=0.27" "pyarrow>=17"

import time
from collections import Counter
from itertools import chain
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq
from datasets import load_dataset
from google.colab import userdata
from huggingface_hub import HfApi, login
from huggingface_hub import login, notebook_login
from tqdm.auto import tqdm


notebook_login()
api = HfApi()
api.create_repo(
    repo_id=DEST_REPO,
    repo_type="dataset",
    private=False,
    exist_ok=True,
)

# 已上传文件用于断点续传。重新运行时会重建分片，但不会重复上传已有分片。
remote_files = set(api.list_repo_files(DEST_REPO, repo_type="dataset"))
work_dir = Path("/content/chinese_corpus_upload")
work_dir.mkdir(parents=True, exist_ok=True)


def cleaned(value):
    """将字段规范为非空字符串。"""
    if value is None:
        return ""
    return str(value).strip()


def tiny_stories_rows():
    dataset = load_dataset(
        "RobinChen2001/TinyStories-Zh-2M",
        split="train",
        streaming=True,
    )
    for example in dataset:
        text = cleaned(example.get("text"))
        if text:
            yield {"text": text, "source": "TinyStories-Zh-2M"}


def wikipedia_rows():
    dataset = load_dataset(
        "wikimedia/wikipedia",
        "20231101.zh",
        split="train",
        streaming=True,
    )
    for example in dataset:
        title = cleaned(example.get("title"))
        body = cleaned(example.get("text"))
        if body:
            text = f"{title}\n\n{body}" if title and not body.startswith(title) else body
            yield {"text": text, "source": "Wikipedia-20231101.zh"}


def zhihu_rows():
    dataset = load_dataset(
        "wangrui6/Zhihu-KOL",
        split="train",
        streaming=True,
    )
    for example in dataset:
        question = cleaned(example.get("INSTRUCTION"))
        answer = cleaned(example.get("RESPONSE"))
        if question and answer:
            yield {
                "text": f"问题：{question}\n回答：{answer}",
                "source": "Zhihu-KOL",
            }


def upload_with_retry(local_path, path_in_repo, attempts=5):
    for attempt in range(1, attempts + 1):
        try:
            api.upload_file(
                path_or_fileobj=str(local_path),
                path_in_repo=path_in_repo,
                repo_id=DEST_REPO,
                repo_type="dataset",
                commit_message=f"Upload {path_in_repo}",
            )
            return
        except Exception:
            if attempt == attempts:
                raise
            wait_seconds = 2 ** attempt
            print(f"上传失败，{wait_seconds} 秒后重试 ({attempt}/{attempts})...")
            time.sleep(wait_seconds)


schema = pa.schema([
    pa.field("text", pa.string()),
    pa.field("source", pa.string()),
])

all_rows = chain(tiny_stories_rows(), wikipedia_rows(), zhihu_rows())
counts = Counter()
buffer = {"text": [], "source": []}
writer = None
local_path = None
repo_path = None
shard_index = 0
rows_in_shard = 0
progress = tqdm(desc="已处理样本", unit=" rows")


def flush_buffer():
    global buffer
    if buffer["text"]:
        table = pa.Table.from_pydict(buffer, schema=schema)
        writer.write_table(table)
        buffer = {"text": [], "source": []}


def open_shard():
    global writer, local_path, repo_path
    filename = f"train-{shard_index:05d}.parquet"
    local_path = work_dir / filename
    repo_path = f"data/{filename}"
    writer = pq.ParquetWriter(
        local_path,
        schema,
        compression="zstd",
        use_dictionary=True,
    )


def close_and_upload_shard():
    global writer
    flush_buffer()
    writer.close()
    writer = None

    if repo_path in remote_files:
        print(f"已存在，跳过上传：{repo_path}")
    else:
        upload_with_retry(local_path, repo_path)
        remote_files.add(repo_path)
        print(f"上传完成：{repo_path}")

    local_path.unlink(missing_ok=True)


try:
    for row in all_rows:
        if writer is None:
            open_shard()

        buffer["text"].append(row["text"])
        buffer["source"].append(row["source"])
        counts[row["source"]] += 1
        rows_in_shard += 1
        progress.update(1)

        if len(buffer["text"]) >= WRITE_BATCH_ROWS:
            flush_buffer()

        if rows_in_shard >= SHARD_ROWS:
            close_and_upload_shard()
            shard_index += 1
            rows_in_shard = 0

    if writer is not None:
        close_and_upload_shard()
finally:
    progress.close()
    if writer is not None:
        writer.close()

# 上传数据集说明。混合来源的许可不同，因此不声明一个统一许可证。
readme = f"""---
language:
- zh
task_categories:
- text-generation
pretty_name: Chinese Small LM Corpus
configs:
- config_name: default
  data_files:
  - split: train
    path: data/train-*.parquet
---

# Chinese Small LM Corpus

用于中文小型语言模型预训练的统一文本语料，字段为：

- `text`：规范化后的训练文本
- `source`：原始数据集名称

## 数据量

| 来源 | 有效样本数 |
|---|---:|
| TinyStories-Zh-2M | {counts['TinyStories-Zh-2M']:,} |
| Wikipedia-20231101.zh | {counts['Wikipedia-20231101.zh']:,} |
| Zhihu-KOL | {counts['Zhihu-KOL']:,} |
| 总计 | {sum(counts.values()):,} |

## 来源与许可

1. `RobinChen2001/TinyStories-Zh-2M`：数据卡标注 MIT；同时应检查英文上游数据及机器翻译来源条款。
2. `wikimedia/wikipedia` (`20231101.zh`)：CC BY-SA 3.0 与 GFDL。
3. `wangrui6/Zhihu-KOL`：原数据卡未声明许可证。

此合并数据集不提供统一的再授权。下载者须分别遵守各来源的许可、署名、隐私与内容使用要求。
"""

api.upload_file(
    path_or_fileobj=readme.encode("utf-8"),
    path_in_repo="README.md",
    repo_id=DEST_REPO,
    repo_type="dataset",
    commit_message="Add dataset card",
)

print("\n全部完成！")
print("数据集地址：", f"https://huggingface.co/datasets/{DEST_REPO}")
print("有效样本数：", dict(counts))
print("总样本数：", sum(counts.values()))

已处理样本: 0 rows [00:00, ? rows/s]

README.md:   0%|          | 0.00/1.90k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pload/train-00000.parquet:   3%|3         |  525kB / 15.5MB            

...

上传完成：data/train-00086.parquet


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pload/train-00087.parquet:   1%|1         |  524kB / 37.9MB            

上传完成：data/train-00087.parquet

全部完成！
数据集地址： https://huggingface.co/datasets/richarddzh/chinese-small-lm-corpus
有效样本数： {'TinyStories-Zh-2M': 1994291, 'Wikipedia-20231101.zh': 1384748, 'Zhihu-KOL': 1002863}
总样本数： 4381902
